# 20.4 深度异常检测 / Advanced (Deep) Anomaly Detection

**中文**:异常检测要解决一个特殊的机器学习问题:**只有(或几乎只有)正常样本,异常极其稀少、形态千变万化,无法穷举**。工业质检(找产品缺陷)、欺诈检测、系统监控、医疗筛查都是如此——你**没法把"所有异常长什么样"教给模型**,只能让它**学透"正常长什么样",凡是偏离正常的就报警**。Part 14 讲过时序异常(STL+MAD),本节讲**高维数据(尤其图像)的深度异常检测**:**自编码器(重构误差)、Deep SVDD(超球)、PaDiM(补丁分布)**。核心思想:**学习正常的紧凑表示,异常无处安放**。
**English**: Anomaly detection solves a special ML problem: **you have (almost) only normal samples; anomalies are extremely rare and endlessly varied, impossible to enumerate**. Industrial inspection (finding defects), fraud detection, system monitoring, medical screening are all like this — you **can't teach the model "what all anomalies look like,"** only let it **learn "what normal looks like" thoroughly and flag anything deviating**. Part 14 covered time-series anomalies (STL+MAD); this section covers **deep anomaly detection for high-dimensional data (especially images)**: **autoencoders (reconstruction error), Deep SVDD (hypersphere), PaDiM (patch distribution)**. The core idea: **learn a compact representation of normal; anomalies have nowhere to fit**.

---

**中文**:三种主流深度方法:
**English**: Three mainstream deep methods:

**中文**:
- **① 自编码器(Autoencoder)重构法**:在正常数据上训练一个自编码器(压缩再重建, Part 13)。它学会了"如何重建正常样本",于是**正常样本重构得很好(误差小),异常样本重构得很差(误差大)**——用**重构误差**当异常分数。最直观、最常用。
  **Autoencoder reconstruction**: train an autoencoder (compress-then-rebuild, Part 13) on normal data. It learns "how to reconstruct normal," so **normal samples reconstruct well (low error) and anomalies reconstruct poorly (high error)** — use **reconstruction error** as the anomaly score. The most intuitive and common.
- **② Deep SVDD(深度支持向量数据描述)**:训练一个网络,把所有正常样本映射进一个**尽可能小的超球**里(最小化嵌入到球心的距离)。**异常样本会落在球外**(离球心远)——用到球心的距离当异常分数。是 One-Class SVM 的深度版。
  **Deep SVDD (Support Vector Data Description)**: train a network to map all normal samples into the **smallest possible hypersphere** (minimize the distance of embeddings to the center). **Anomalies fall outside** (far from the center) — use distance-to-center as the score. The deep version of One-Class SVM.
- **③ PaDiM(补丁分布建模)**:专为**图像缺陷检测**(如 MVTec 工业数据集)。用预训练 CNN 提取每个**图像补丁(patch)** 的特征,对正常图像的每个位置的补丁特征拟合一个高斯分布;测试时用 **马氏距离(Mahalanobis)** 衡量新补丁偏离正常分布多远,**能精确定位缺陷在图像的哪个位置**。
  **PaDiM (Patch Distribution Modeling)**: designed for **image defect detection** (e.g. the MVTec industrial dataset). Use a pretrained CNN to extract features of each **image patch**, fit a Gaussian to normal patch features at each position; at test time use the **Mahalanobis distance** to measure how far a new patch deviates, **precisely localizing the defect in the image**.

> 💡 **面试速查 / Interview cheat-sheet（★★ 质检/风控必考）**
> **中文**:异常检测=**只学正常、报警偏离**(异常稀少、无法穷举, 是单类/无监督问题)。深度方法:①**自编码器**:正常上训, 异常**重构误差大**(最常用);②**Deep SVDD**:把正常映进最小超球, 异常在球外(距球心远), One-Class SVM 深度版;③**PaDiM/PatchCore**:图像缺陷检测, 预训练特征+补丁高斯+马氏距离, **能定位缺陷位置**(工业质检SOTA)。评估:训练**无异常标签**, 测试用 **AUC/AUPRC**(异常少, PR 曲线更实在)。**坑**:①正常里混入异常会毒化模型;②"正常"分布漂移(概念漂移);③自编码器可能"泛化过头"把异常也重构好;④阈值怎么定(靠正常样本误差分布的分位数)。经典浅层法:Isolation Forest、LOF、One-Class SVM、Elliptic Envelope。
> **English**: Anomaly detection = **learn only normal, flag deviations** (anomalies rare, unenumerable — a one-class/unsupervised problem). Deep methods: ① **autoencoder**: train on normal, anomalies have **high reconstruction error** (most common); ② **Deep SVDD**: map normal into the smallest hypersphere, anomalies outside (far from center), the deep One-Class SVM; ③ **PaDiM/PatchCore**: image defect detection, pretrained features + patch Gaussians + Mahalanobis, **localizes the defect** (industrial-inspection SOTA). Evaluation: training has **no anomaly labels**, test with **AUC/AUPRC** (anomalies are rare, PR curves are more meaningful). **Pitfalls**: ① anomalies contaminating "normal" training poison the model; ② "normal" distribution drift (concept drift); ③ autoencoders may "over-generalize" and reconstruct anomalies too well; ④ threshold-setting (use quantiles of normal-sample error). Classic shallow methods: Isolation Forest, LOF, One-Class SVM, Elliptic Envelope.


In [ ]:

# ============================================================
# 单类设定:一个数字当"正常", 其余当"异常" / one-class setup: one digit normal, others anomalies
# 中文:只用数字"1"的图片训练(=只见过正常)。测试集混入其他数字(=异常)。模型没见过任何异常。
# English: train only on digit "1" (only sees normal). Test set mixes in other digits (anomalies). The model never sees an anomaly.
# ============================================================
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, os, matplotlib.pyplot as plt
from torchvision import datasets
torch.manual_seed(0); np.random.seed(0)
root=os.path.expanduser("~/.cache/dsfs_cv")
mnist=datasets.MNIST(root, train=True, download=False)
X=(mnist.data.float()/255.).numpy(); y=mnist.targets.numpy()
NORMAL=1
Xtrain=X[y==NORMAL][:3000]                                   # 训练:只有正常(数字1)/ train: only normal
te_norm=X[y==NORMAL][3000:3500]; te_anom=X[y!=NORMAL][:500]  # 测试:正常 + 异常(其他数字)/ test: normal + anomalies
Xte=np.concatenate([te_norm,te_anom]); yte=np.r_[np.zeros(len(te_norm)),np.ones(len(te_anom))]  # 1=异常/anomaly
Xtr_t=torch.tensor(Xtrain).view(-1,784); Xte_t=torch.tensor(Xte).view(-1,784)
def auc(scores,labels):
    o=np.argsort(scores); r=np.empty(len(scores)); r[o]=np.arange(len(scores))
    npos=labels.sum(); nneg=len(labels)-npos
    return (r[labels==1].sum()-npos*(npos-1)/2)/(npos*nneg)
print(f"正常类=数字{NORMAL}, 训练 {len(Xtrain)} 张(全正常), 测试 {len(te_norm)} 正常 + {len(te_anom)} 异常")


**中文**:**① 自编码器重构法**。在数字 1 上训练自编码器,它会学得"如何把 1 压缩再重建得很好"。测试时,数字 1 重构误差小、其他数字重构误差大——用重构误差当异常分数。
**English**: **① Autoencoder reconstruction**. Train an autoencoder on digit 1; it learns "how to compress and rebuild 1 well." At test, digit 1 has low reconstruction error and other digits high — use reconstruction error as the anomaly score.


In [ ]:

# ============================================================
# ① 自编码器异常检测 / autoencoder anomaly detection
# ============================================================
class AE(nn.Module):
    def __init__(s):
        super().__init__()
        s.enc=nn.Sequential(nn.Linear(784,128),nn.ReLU(),nn.Linear(128,16))   # 压缩到16维 / bottleneck
        s.dec=nn.Sequential(nn.Linear(16,128),nn.ReLU(),nn.Linear(128,784),nn.Sigmoid())
    def forward(s,x): z=s.enc(x); return s.dec(z), z
ae=AE(); opt=torch.optim.Adam(ae.parameters(),1e-3)
t0=time.time()
for e in range(20):
    perm=torch.randperm(len(Xtr_t))
    for b in range(0,len(Xtr_t),128):
        x=Xtr_t[perm[b:b+128]]; xh,_=ae(x); loss=F.mse_loss(xh,x)   # 只学重建正常 / reconstruct normal
        opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad():
    xh,_=ae(Xte_t); recon_err=((xh-Xte_t)**2).mean(1).numpy()      # 重构误差=异常分数 / anomaly score
auc_ae=auc(recon_err, yte)
print(f"自编码器异常检测 AUC / autoencoder AUC: {auc_ae:.3f}  ({time.time()-t0:.0f}s)")
print(f"正常样本平均重构误差 {recon_err[yte==0].mean():.4f} vs 异常 {recon_err[yte==1].mean():.4f} (异常大得多)")


**中文**:**② Deep SVDD**。训练一个网络,把所有正常样本(数字1)的嵌入**挤进一个小球**里(最小化到球心的距离)。异常样本嵌入后会落在球外,离球心远——用到球心的距离当异常分数。
**English**: **② Deep SVDD**. Train a network to **squeeze all normal (digit 1) embeddings into a small ball** (minimize distance to the center). Anomalies' embeddings fall outside, far from the center — use distance-to-center as the anomaly score.


In [ ]:

# ============================================================
# ② Deep SVDD / one-class hypersphere
# ============================================================
class Encoder(nn.Module):
    def __init__(s): super().__init__(); s.net=nn.Sequential(nn.Linear(784,128),nn.ReLU(),nn.Linear(128,16,bias=False))
    def forward(s,x): return s.net(x)
enc=Encoder()
with torch.no_grad(): center=enc(Xtr_t).mean(0)              # 球心=初始嵌入均值 / center = mean embedding
opt=torch.optim.Adam(enc.parameters(),1e-3)
for e in range(20):
    perm=torch.randperm(len(Xtr_t))
    for b in range(0,len(Xtr_t),128):
        z=enc(Xtr_t[perm[b:b+128]]); loss=((z-center)**2).sum(1).mean()   # 挤向球心 / pull toward center
        opt.zero_grad(); loss.backward(); opt.step()
with torch.no_grad(): svdd_score=((enc(Xte_t)-center)**2).sum(1).numpy()   # 到球心距离=异常分数 / distance to center
auc_svdd=auc(svdd_score, yte)
print(f"Deep SVDD 异常检测 AUC / Deep SVDD AUC: {auc_svdd:.3f}")
print(f"{'方法/method':<20}{'AUC':>8}")
print(f"{'自编码器 AE':<20}{auc_ae:>8.3f}")
print(f"{'Deep SVDD':<20}{auc_svdd:>8.3f}")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,3,figsize=(16,4.5))
# ① 重构误差分布:正常 vs 异常 / reconstruction error distribution
ax[0].hist(recon_err[yte==0],bins=40,alpha=0.6,color="#4C72B0",label="正常(数字1)",density=True)
ax[0].hist(recon_err[yte==1],bins=40,alpha=0.6,color="#C44E52",label="异常(其他数字)",density=True)
ax[0].set_title("重构误差:正常小、异常大 / reconstruction error"); ax[0].set_xlabel("重构误差(异常分数)"); ax[0].legend(fontsize=9)
# ② 重建示例:正常重建好, 异常重建差(被"抹成1")/ reconstruction examples
with torch.no_grad():
    ex_n=Xte_t[yte==0][:1]; ex_a=Xte_t[yte==1][3:4]
    rn,_=ae(ex_n); ra,_=ae(ex_a)
for i,(orig,rec,lab) in enumerate([(ex_n,rn,"正常输入"),(rn,rn,"正常重建"),(ex_a,ra,"异常输入"),(ra,ra,"异常重建")]):
    a=fig.add_subplot(2,6,[3,4,9,10][i]); a.imshow(orig.view(28,28) if i%2==0 else rec.view(28,28),cmap="gray")
    a.set_title(lab,fontsize=8); a.axis("off")
ax[1].axis("off"); ax[1].set_title("重建:异常被'抹成正常', 误差大 / anomaly rebuilt as normal",fontsize=10)
# ③ AUC 对比 / AUC comparison
ax[2].bar(["自编码器\nAE","Deep\nSVDD"],[auc_ae,auc_svdd],color=["#4C72B0","#55A868"])
ax[2].axhline(0.5,ls="--",color="gray",label="随机 0.5"); ax[2].set_ylim(0,1.05)
for i,v in enumerate([auc_ae,auc_svdd]): ax[2].text(i,v,f"{v:.3f}",ha="center",va="bottom")
ax[2].set_title("异常检测 AUC(都远超随机)/ anomaly AUC"); ax[2].set_ylabel("AUC"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig("/tmp/adv04_viz.png",dpi=80); plt.show()
print("两种深度方法都只见过正常样本, 却能高精度识别从没见过的异常(AUC>0.99)")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **"只学正常"就能抓异常**:两个模型**训练时一张异常都没见过**(只见过数字1),测试时却能以 AUC>0.99 识别出各种从没见过的异常数字。这正是异常检测与普通分类的根本区别——**你无法收集"所有异常"的样本**(缺陷千变万化、欺诈手段日新月异),所以只能反过来:**把正常学到极致,凡是模型"看不懂/重建不好/放不进正常的框"的就是异常**。
2. **两种方法的直觉互补**:自编码器说"**异常我重建不好**"(重构误差大);Deep SVDD 说"**异常离我的正常中心太远**"(嵌入在球外)。看中图的重建示例——一个异常数字被自编码器强行"抹成了数字1"的样子(因为它只会重建1),所以重构误差很大。这个可视化把"重构法为什么有效"讲得一清二楚。
3. **诚实的陷阱**:①**本例太干净了(AUC>0.99)**——因为数字1的形态非常独特、简单。真实工业缺陷检测难得多(缺陷可能只是产品上一个微小划痕,占几个像素),这时 PaDiM/PatchCore 这种**基于补丁 + 预训练特征 + 定位**的方法才是 SOTA;②**训练数据必须干净**——如果"正常"里混进了异常,模型会把异常也学成正常,检测失效(数据清洗关键);③**自编码器可能泛化过头**——如果模型容量太大,它可能连异常也重建得不错,反而检测不出(要控制瓶颈维度);④**概念漂移**——"正常"会随时间变(新款产品),模型要定期重训;⑤评估用 AUC 还不够,异常极少时 **AUPRC(PR曲线下面积)** 更能反映真实性能。

**English**:
1. **"Learning only normal" catches anomalies**: both models **saw not a single anomaly during training** (only digit 1), yet detect all sorts of never-seen anomaly digits at AUC>0.99. This is the fundamental difference from ordinary classification — **you can't collect samples of "all anomalies"** (defects vary endlessly, fraud evolves daily), so you invert it: **learn normal to the extreme; whatever the model "can't understand / rebuilds poorly / can't fit into the normal box" is an anomaly**.
2. **The two methods' intuitions are complementary**: the autoencoder says "**I can't rebuild the anomaly**" (high reconstruction error); Deep SVDD says "**the anomaly is too far from my normal center**" (embedding outside the ball). See the middle plot's reconstruction examples — an anomaly digit is forcibly "smudged into a digit 1" by the autoencoder (which only knows how to rebuild 1s), hence high error. This visualization makes "why reconstruction works" crystal clear.
3. **Honest pitfalls**: ① **this example is too clean (AUC>0.99)** — because digit 1 is very distinctive and simple. Real industrial defect detection is much harder (a defect may be a tiny scratch of a few pixels), where **patch-based + pretrained-feature + localizing** methods like PaDiM/PatchCore are SOTA; ② **training data must be clean** — if anomalies contaminate "normal," the model learns them as normal and detection fails (data cleaning is key); ③ **autoencoders may over-generalize** — with too much capacity they rebuild anomalies well too, missing them (control the bottleneck dimension); ④ **concept drift** — "normal" shifts over time (new products), needing periodic retraining; ⑤ AUC alone isn't enough — when anomalies are very rare, **AUPRC (area under PR curve)** better reflects real performance.

> 💼 **实战视角 / Practical angle**
> **中文**:深度异常检测的主战场:①**工业质检**(找产品缺陷, MVTec 数据集, PaDiM/PatchCore/SimpleNet 是SOTA, 能定位缺陷像素);②**欺诈/风控**(异常交易, 常配 Isolation Forest 等浅层法);③**系统/APM 监控**(指标异常, 结合 Part 14 时序);④**医疗影像筛查**(找异常病灶);⑤**网络入侵检测**。选型:表格数据先试 **Isolation Forest/LOF**(简单强); 图像用 **PatchCore/PaDiM**(预训练特征 + 定位); 时序用重构/预测误差。落地要点:①**训练集要纯正常**(先清洗);②阈值用正常误差分布的分位数定;③关注**能否定位**异常(工业质检要指出缺陷在哪);④持续监控概念漂移。面试金句:*"异常检测只学正常、报警偏离(异常无法穷举); 自编码器用重构误差、Deep SVDD 用到超球心距离、PaDiM 用补丁高斯+马氏距离并能定位; 训练无异常标签, 评估用 AUC/AUPRC, 关键坑是正常数据被异常污染和概念漂移。"*
> **English**: Deep anomaly detection's arenas: ① **industrial inspection** (find product defects, MVTec dataset; PaDiM/PatchCore/SimpleNet are SOTA and localize defect pixels); ② **fraud/risk** (anomalous transactions, often with shallow methods like Isolation Forest); ③ **system/APM monitoring** (metric anomalies, with Part 14 time series); ④ **medical-image screening** (find abnormal lesions); ⑤ **network intrusion detection**. Selection: for tabular start with **Isolation Forest/LOF** (simple and strong); for images use **PatchCore/PaDiM** (pretrained features + localization); for time series use reconstruction/forecast error. Deployment keys: ① **the training set must be pure normal** (clean it first); ② set thresholds by quantiles of normal-error; ③ care about **localization** (inspection must point to where the defect is); ④ monitor concept drift continuously. Interview line: *"Anomaly detection learns only normal and flags deviations (anomalies are unenumerable); autoencoders use reconstruction error, Deep SVDD distance-to-hypersphere-center, PaDiM patch Gaussians + Mahalanobis with localization; training has no anomaly labels, evaluate with AUC/AUPRC; key pitfalls are anomaly-contaminated normal data and concept drift."*

---
### 小结 / Summary
- **中文**:异常检测只学正常、报警偏离(异常稀少无法穷举, 单类问题); 训练无异常标签, 评估用 AUC/AUPRC。
- **English**: Anomaly detection learns only normal and flags deviations (rare, unenumerable — one-class); training has no anomaly labels, evaluate with AUC/AUPRC.
- **中文**:自编码器(重构误差大)、Deep SVDD(超球外)、PaDiM(补丁高斯+马氏距离, 图像缺陷定位)。
- **English**: Autoencoder (high reconstruction error), Deep SVDD (outside hypersphere), PaDiM (patch Gaussians + Mahalanobis, localizes image defects).
- **中文**:坑:正常被异常污染、自编码器泛化过头、概念漂移; 图像质检 PatchCore/PaDiM 是 SOTA。
- **English**: Pitfalls: anomaly-contaminated normal, autoencoder over-generalization, concept drift; PatchCore/PaDiM are SOTA for image inspection.
